In [4]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

def clean_news_text(text):
    """Removes source bias (Reuters datelines) and cleans whitespace."""
    if not isinstance(text, str): return ""
    # Remove datelines like "CITY (Reuters) - "
    text = re.sub(r'^.*?\(Reuters\)\s*-\s*', '', text)
    # Remove remaining "Reuters" mentions
    text = re.sub(r'\bReuters\b', '', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()

def extract_meta_features(df):
    """Extracts numerical features based on punctuation and capitalization."""
    # Features from Title
    df['title_len'] = df['title'].apply(lambda x: len(str(x).split()))
    df['title_excl'] = df['title'].apply(lambda x: str(x).count('!'))
    df['title_caps'] = df['title'].apply(lambda x: sum(1 for w in str(x).split() if w.isupper() and len(w) > 2))
    
    # Features from Text
    df['text_excl'] = df['text'].apply(lambda x: str(x).count('!'))
    df['text_ques'] = df['text'].apply(lambda x: str(x).count('?'))
    df['text_caps'] = df['text'].apply(lambda x: sum(1 for w in str(x).split() if w.isupper() and len(w) > 2))
    
    return df

# 1. Load Data
df = pd.read_csv('train.csv', sep=';')

# 2. Extract Numerical Features
df = extract_meta_features(df)

# 3. Clean Text for TF-IDF (removing 'Reuters' bias)
df['clean_content'] = df['title'] + " " + df['text'].apply(clean_news_text)

# 4. TF-IDF Vectorization (Textual Features)
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['clean_content'])

# 5. Scale Numerical Features
meta_cols = ['title_len', 'title_excl', 'title_caps', 'text_excl', 'text_ques', 'text_caps']
scaler = StandardScaler()
X_meta = scaler.fit_transform(df[meta_cols])

# 6. Combine all features into one matrix
# This stacks the sparse TF-IDF matrix with the scaled numerical features
X_final = hstack([X_tfidf, X_meta])
df = df.drop(columns=['Unnamed: 0'])
print(f"Feature extraction complete.")
print(f"Total features: {X_final.shape[1]} (5000 TF-IDF + {len(meta_cols)} Meta features)")

df.to_csv('train_cleaned.csv', index=False)

Feature extraction complete.
Total features: 5006 (5000 TF-IDF + 6 Meta features)
